In [2]:
# ================================
# 1. Install Dependencies
# ================================
!pip install transformers datasets scikit-learn

# ================================
# 2. Imports
# ================================
import json
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from tqdm import tqdm

# For auto download (Colab)
from google.colab import files

# ================================
# 3. Load Dataset
# ================================
file_path = "/content/OS_relations_fixed.json"

with open(file_path, "r") as f:
    data = json.load(f)

print("Total samples:", len(data))

dataset = Dataset.from_list(data)

# ================================
# 4. Label Setup
# ================================
LABELS = ["used_for", "based_on", "implements", "part_of", "improves", "no_relation"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

# ================================
# 5. Load SciBERT
# ================================
model_name = "allenai/scibert_scivocab_uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
base_model.to(device)

# ================================
# 6. Add Classification Head
# ================================
class SciBERTClassifier(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()
        self.bert = base_model
        self.classifier = nn.Linear(base_model.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids   # 👈 FIX
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_output)
        return logits

model = SciBERTClassifier(base_model, len(LABELS)).to(device)

# ⚠️ NOTE:
# This is ZERO-SHOT-style (no training), so weights are random for classifier.
# For real performance → you should fine-tune (tell me if you want that).

# ================================
# 7. Prediction Function
# ================================
def predict(example):

    text = example["input"]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs)

    pred_id = torch.argmax(logits, dim=1).item()
    pred_label = id2label[pred_id]

    return pred_label

# ================================
# 8. Run Predictions + Store Triplets
# ================================
y_true = []
y_pred = []
results = []

for example in tqdm(dataset):
    text = example["input"]
    gt = example["output"].strip()

    pred = predict(example)

    y_true.append(gt)
    y_pred.append(pred)

    results.append({
        "input": text,
        "ground_truth": gt,
        "predicted": pred
    })

# ================================
# 9. Save JSON
# ================================
output_file = "/content/scibert_zero_shot_triplets.json"

with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\n✅ Triplets saved to: {output_file}")

# Auto-download
files.download(output_file)

# ================================
# 10. Metrics
# ================================
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print("\n===== SCIBERT ZERO-SHOT RESULTS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\n===== Classification Report =====")
print(classification_report(y_true, y_pred))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y_true, y_pred))

# ================================
# 11. Show Predictions
# ================================
for i in range(min(5, len(dataset))):
    print("\n--- Example ---")
    print("Input :", dataset[i]["input"])
    print("GT    :", y_true[i])
    print("Pred  :", y_pred[i])

Total samples: 1957


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

...


✅ Triplets saved to: /content/scibert_zero_shot_triplets.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


===== SCIBERT ZERO-SHOT RESULTS =====
Accuracy : 0.1574
Precision: 0.2583
Recall   : 0.1574
F1 Score : 0.1804

===== Classification Report =====
              precision    recall  f1-score   support

    based_on       0.02      0.13      0.04        70
  implements       0.07      0.20      0.10        99
    improves       0.01      0.21      0.02        34
 no_relation       0.47      0.29      0.36       915
     part_of       0.12      0.01      0.02       558
    used_for       0.00      0.00      0.00       281

    accuracy                           0.16      1957
   macro avg       0.11      0.14      0.09      1957
weighted avg       0.26      0.16      0.18      1957


===== Confusion Matrix =====
[[  9   9  14  11  27   0]
 [  6  20  23  47   3   0]
 [ 17   2   7   7   1   0]
 [198 128 313 266  10   0]
 [158  89 183 122   6   0]
 [ 44  47  74 113   3   0]]

--- Example ---
Input : Sentence: FAT (File Allocation Table): An older file system used by older versions of Windows

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
